# Training and Test Data Splits in Machine Learning

This notebook explores the fundamental concept of splitting datasets for training and evaluating machine learning models. We'll cover various splitting techniques, their implementations, and best practices.

## Import Required Libraries

We'll need various Python libraries for data manipulation, visualization, and implementing data splitting techniques.

In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import scikit-learn modules for data splitting and example datasets
from sklearn.model_selection import (
    train_test_split, 
    KFold, 
    StratifiedKFold, 
    cross_val_score,
    RepeatedKFold,
    TimeSeriesSplit,
    StratifiedShuffleSplit
)
from sklearn.datasets import load_iris, load_boston, load_diabetes, load_breast_cancer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Set random seed for reproducibility
np.random.seed(42)

# Set up some display parameters
plt.style.use('seaborn-whitegrid')
sns.set_theme(style="whitegrid")

## Introduction to Data Splitting

In machine learning, we typically divide our dataset into separate portions:

- **Training set**: Used to train the model
- **Validation set**: Used for tuning hyperparameters and making design choices
- **Test set**: Used to evaluate the final model performance

This separation is crucial because:
1. It helps assess the model's ability to generalize to new, unseen data
2. It prevents overfitting by evaluating on data not used in training
3. It provides an unbiased estimate of the model's performance

Let's load some example datasets to demonstrate these concepts:

In [ ]:
# Load sample datasets
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
iris_feature_names = iris.feature_names
iris_target_names = iris.target_names

# Convert to pandas DataFrame for easier handling
iris_df = pd.DataFrame(X_iris, columns=iris_feature_names)
iris_df['target'] = y_iris
iris_df['species'] = iris_df['target'].map({
    0: iris_target_names[0],
    1: iris_target_names[1],
    2: iris_target_names[2]
})

# Display the first few rows
print("Iris Dataset Preview:")
print(f"Total samples: {len(iris_df)}")
print(f"Features: {iris_feature_names}")
print(f"Target classes: {iris_target_names}")
iris_df.head()

In [ ]:
# Load a regression dataset
try:
    boston = load_boston()
    X_boston, y_boston = boston.data, boston.target
    boston_feature_names = boston.feature_names
    
    # Convert to pandas DataFrame
    boston_df = pd.DataFrame(X_boston, columns=boston_feature_names)
    boston_df['target'] = y_boston
    
    print("\nBoston Housing Dataset Preview:")
    print(f"Total samples: {len(boston_df)}")
    print(f"Features: {boston_feature_names}")
    boston_df.head()
except:
    # The Boston dataset might be deprecated, so we'll use diabetes as backup
    diabetes = load_diabetes()
    X_diabetes, y_diabetes = diabetes.data, diabetes.target
    diabetes_feature_names = diabetes.feature_names
    
    # Convert to pandas DataFrame
    diabetes_df = pd.DataFrame(X_diabetes, columns=diabetes_feature_names)
    diabetes_df['target'] = y_diabetes
    
    print("\nDiabetes Dataset Preview:")
    print(f"Total samples: {len(diabetes_df)}")
    print(f"Features: {diabetes_feature_names}")
    diabetes_df.head()

## Basic Train-Test Split

The simplest approach to data splitting is dividing the data into training and test sets using `train_test_split` from scikit-learn. This function allows us to:

- Specify the proportion of data to use for testing (via `test_size` parameter)
- Set a random seed for reproducibility (via `random_state` parameter)
- Choose whether to shuffle the data (via `shuffle` parameter)

Let's implement a basic split and examine the outputs:

In [ ]:
# Basic train-test split with the Iris dataset
X = iris_df.drop(['target', 'species'], axis=1)
y = iris_df['target']

# Split with 80% training data and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print the shapes of resulting datasets
print(f"Original dataset shape: {X.shape}")
print(f"Training set shape: {X_train.shape} - {X_train.shape[0]/X.shape[0]:.1%} of original data")
print(f"Test set shape: {X_test.shape} - {X_test.shape[0]/X.shape[0]:.1%} of original data")

In [ ]:
# Visualize the train-test split using a pair plot
# First, create a copy of the dataframe
iris_split_df = iris_df.copy()

# Add a column to indicate if a sample is in the training or test set
all_indices = set(range(len(iris_df)))
test_indices = set(X_test.index)
train_indices = all_indices - test_indices

iris_split_df['split'] = 'train'
iris_split_df.loc[list(test_indices), 'split'] = 'test'

# Create a pair plot to visualize the split
plt.figure(figsize=(12, 8))
sns.pairplot(
    iris_split_df, 
    hue='species',
    style='split',
    diag_kind='kde',
    plot_kws={'alpha': 0.7},
    height=2.5
)
plt.suptitle('Visualization of Train-Test Split on Iris Dataset', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### Evaluating Model Performance with Different Split Ratios

The choice of split ratio can impact model performance. Let's explore different train-test splits and evaluate a simple model:

In [ ]:
# Test different train-test split ratios
test_sizes = [0.1, 0.2, 0.3, 0.4, 0.5]
accuracy_scores = []

for test_size in test_sizes:
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # Create and train a simple model
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)
    
    # Evaluate the model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_scores.append(accuracy)
    
    print(f"Test size: {test_size:.1f} ({len(X_test)} samples) - Accuracy: {accuracy:.4f}")

# Visualize the results
plt.figure(figsize=(10, 6))
plt.plot(test_sizes, accuracy_scores, marker='o', linestyle='-', linewidth=2, markersize=10)
plt.xlabel('Test Set Size (proportion)', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.title('Model Accuracy with Different Train-Test Split Ratios', fontsize=14)
plt.xticks(test_sizes)
plt.grid(True)
plt.show()

## Cross-Validation

Basic train-test splits have limitations - the evaluation depends heavily on which data points happen to be in the test set. **Cross-validation** addresses this by:

1. Dividing the data into k equally sized "folds"
2. Training the model k times, each time using a different fold as the test set and the remaining folds as training data
3. Averaging the performance across all k iterations

This approach provides a more robust estimate of model performance.

In [ ]:
# Implement k-fold cross-validation
k_values = [3, 5, 10]
model = LogisticRegression(max_iter=200)

plt.figure(figsize=(14, 8))

for i, k in enumerate(k_values):
    # Create a KFold cross-validator
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    # Keep track of all fold scores
    fold_scores = []
    
    # Plot the data splits
    plt.subplot(1, 3, i+1)
    plt.title(f"{k}-Fold Cross-Validation")
    
    # Create a color map for the folds
    cmap = plt.cm.tab10
    
    for j, (train_idx, test_idx) in enumerate(kf.split(X)):
        # Train the model
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_test_fold)
        score = accuracy_score(y_test_fold, y_pred)
        fold_scores.append(score)
        
        # Visualize the fold
        plt.scatter(range(len(X)), [j+1] * len(X), c='lightgray', s=10, alpha=0.3)
        plt.scatter(test_idx, [j+1] * len(test_idx), c=cmap(j % 10), s=40, label=f"Fold {j+1}")
    
    # Print the fold scores
    print(f"\n{k}-Fold Cross-Validation:")
    for j, score in enumerate(fold_scores):
        print(f"  Fold {j+1}: {score:.4f}")
    print(f"  Mean Accuracy: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
    
    plt.yticks(range(1, k+1))
    plt.xlabel("Sample index")
    plt.ylabel("Fold")
    if i == 2:  # Only show legend for the last plot
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5)

plt.tight_layout()
plt.show()

In [ ]:
# Using cross_val_score for simpler execution
cv_results = {}

for k in k_values:
    scores = cross_val_score(
        LogisticRegression(max_iter=200),
        X, y,
        cv=k,
        scoring='accuracy'
    )
    cv_results[k] = scores

# Visualize the distribution of scores
plt.figure(figsize=(10, 6))
positions = range(1, len(k_values) + 1)

# Create box plot
box = plt.boxplot(
    [cv_results[k] for k in k_values],
    positions=positions,
    patch_artist=True,
    notch=True,
    widths=0.4
)

# Customize box colors
colors = ['lightblue', 'lightgreen', 'lightsalmon']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Add scatter plot of individual points
for i, k in enumerate(k_values):
    plt.scatter([i+1] * len(cv_results[k]), cv_results[k], 
                alpha=0.7, s=50, color='darkblue', zorder=3)
    
plt.xticks(positions, [f"{k}-Fold" for k in k_values])
plt.ylabel('Accuracy Score', fontsize=12)
plt.title('Distribution of Cross-Validation Scores', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, k in enumerate(k_values):
    mean_score = np.mean(cv_results[k])
    plt.text(i+1, 0.92, f"Mean: {mean_score:.3f}", 
             ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Repeated K-Fold Cross-Validation

To get even more robust estimates, we can use Repeated K-Fold Cross-Validation, which performs K-Fold CV multiple times with different random splits.

In [ ]:
# Repeated K-Fold Cross-Validation
n_repeats = 3
k = 5

# Create the repeated k-fold cross-validator
repeated_kf = RepeatedKFold(n_splits=k, n_repeats=n_repeats, random_state=42)

# Train and evaluate the model
model = LogisticRegression(max_iter=200)
repeated_scores = cross_val_score(model, X, y, cv=repeated_kf, scoring='accuracy')

# Reshape the scores to visualize by repeat
scores_by_repeat = repeated_scores.reshape(n_repeats, k)

# Calculate statistics
mean_by_repeat = np.mean(scores_by_repeat, axis=1)
std_by_repeat = np.std(scores_by_repeat, axis=1)

# Print results
print(f"Repeated {k}-Fold Cross-Validation ({n_repeats} repeats):")
for i in range(n_repeats):
    fold_scores = scores_by_repeat[i]
    print(f"  Repeat {i+1}: Mean={mean_by_repeat[i]:.4f}, Std={std_by_repeat[i]:.4f}")
    for j, score in enumerate(fold_scores):
        print(f"    Fold {j+1}: {score:.4f}")

print(f"\nOverall: Mean={np.mean(repeated_scores):.4f}, Std={np.std(repeated_scores):.4f}")

# Visualize the results
plt.figure(figsize=(12, 6))

# Plot individual fold scores
for i in range(n_repeats):
    plt.plot(range(1, k+1), scores_by_repeat[i], 'o-', 
             alpha=0.7, label=f'Repeat {i+1}')

# Plot the overall mean
plt.axhline(y=np.mean(repeated_scores), color='r', linestyle='--', 
            label=f'Overall Mean: {np.mean(repeated_scores):.4f}')

plt.xlabel('Fold Number')
plt.ylabel('Accuracy Score')
plt.title(f'Repeated {k}-Fold Cross-Validation Scores')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xticks(range(1, k+1))
plt.tight_layout()
plt.show()

## Stratified Sampling

When dealing with classification problems, especially with imbalanced classes, it's important to maintain the same class distribution in both training and test sets. **Stratified sampling** ensures that the proportion of samples from each class remains the same across splits.

In [ ]:
# Load a dataset with imbalanced classes (breast cancer)
breast_cancer = load_breast_cancer()
X_cancer, y_cancer = breast_cancer.data, breast_cancer.target

# Check class distribution
unique_classes, class_counts = np.unique(y_cancer, return_counts=True)
class_distribution = dict(zip(
    [breast_cancer.target_names[i] for i in unique_classes], 
    class_counts
))

print("Class distribution in the breast cancer dataset:")
for class_name, count in class_distribution.items():
    print(f"  {class_name}: {count} samples ({count/len(y_cancer):.1%})")

# Compare regular vs stratified sampling
plt.figure(figsize=(15, 6))

# Regular train_test_split
plt.subplot(1, 2, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42
)

train_class_counts = np.unique(y_train, return_counts=True)[1]
test_class_counts = np.unique(y_test, return_counts=True)[1]

# Calculate percentages
train_pct = train_class_counts / len(y_train) * 100
test_pct = test_class_counts / len(y_test) * 100

# Plot regular split
x = np.arange(len(unique_classes))
width = 0.35
plt.bar(x - width/2, train_pct, width, label='Train Set')
plt.bar(x + width/2, test_pct, width, label='Test Set')
plt.xlabel('Class')
plt.ylabel('Percentage (%)')
plt.title('Regular Train-Test Split')
plt.xticks(x, breast_cancer.target_names)
plt.legend()

# Stratified train_test_split
plt.subplot(1, 2, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

train_class_counts = np.unique(y_train, return_counts=True)[1]
test_class_counts = np.unique(y_test, return_counts=True)[1]

# Calculate percentages
train_pct = train_class_counts / len(y_train) * 100
test_pct = test_class_counts / len(y_test) * 100

# Plot stratified split
plt.bar(x - width/2, train_pct, width, label='Train Set')
plt.bar(x + width/2, test_pct, width, label='Test Set')
plt.xlabel('Class')
plt.ylabel('Percentage (%)')
plt.title('Stratified Train-Test Split')
plt.xticks(x, breast_cancer.target_names)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Using StratifiedKFold for cross-validation with imbalanced classes
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
regular_kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Function to evaluate and compare CV strategies
def compare_cv_strategies(X, y, cv_strategies, model):
    results = {}
    
    for name, cv in cv_strategies.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
        results[name] = scores
    
    return results

# Compare regular K-Fold with Stratified K-Fold
cv_strategies = {
    'Regular K-Fold': regular_kf,
    'Stratified K-Fold': stratified_kf
}

model = LogisticRegression(max_iter=200)
cv_results = compare_cv_strategies(X_cancer, y_cancer, cv_strategies, model)

# Visualize the comparison
plt.figure(figsize=(10, 6))
colors = ['lightblue', 'lightgreen']
box = plt.boxplot(
    [cv_results[name] for name in cv_strategies.keys()],
    patch_artist=True,
    notch=True,
    labels=list(cv_strategies.keys())
)

# Customize box colors
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Add individual points
for i, name in enumerate(cv_strategies.keys()):
    plt.scatter([i+1] * len(cv_results[name]), cv_results[name], 
                alpha=0.7, s=50, color='darkblue')
    
    # Print statistics
    mean_score = np.mean(cv_results[name])
    std_score = np.std(cv_results[name])
    print(f"{name}: Mean={mean_score:.4f}, Std={std_score:.4f}")
    
    # Add mean to the plot
    plt.text(i+1, min(cv_results[name])-0.01, f"μ={mean_score:.3f}", 
             ha='center', fontweight='bold')

plt.ylabel('Accuracy Score')
plt.title('Comparison of K-Fold vs Stratified K-Fold')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### Stratified Shuffle Split

`StratifiedShuffleSplit` combines the benefits of random sampling and stratification. Let's explore this approach:

In [ ]:
# Create a highly imbalanced synthetic dataset for demonstration
from sklearn.datasets import make_classification

# Create synthetic imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42
)

# Check class distribution
unique_classes, class_counts = np.unique(y_imb, return_counts=True)
print("Synthetic imbalanced dataset class distribution:")
for i, count in zip(unique_classes, class_counts):
    print(f"  Class {i}: {count} samples ({count/len(y_imb):.1%})")

# Create a StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(sss.split(X_imb, y_imb))

X_train, X_test = X_imb[train_idx], X_imb[test_idx]
y_train, y_test = y_imb[train_idx], y_imb[test_idx]

# Verify class distribution is maintained
train_class_dist = np.unique(y_train, return_counts=True)[1] / len(y_train)
test_class_dist = np.unique(y_test, return_counts=True)[1] / len(y_test)
original_class_dist = class_counts / len(y_imb)

# Plot the results
plt.figure(figsize=(12, 10))

# Plot the dataset with train/test split
plt.subplot(2, 1, 1)
plt.scatter(X_imb[y_imb==0, 0], X_imb[y_imb==0, 1], 
            c='blue', marker='o', s=20, alpha=0.5, label='Class 0')
plt.scatter(X_imb[y_imb==1, 0], X_imb[y_imb==1, 1], 
            c='red', marker='^', s=80, alpha=0.5, label='Class 1')
plt.title('Synthetic Imbalanced Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()

# Plot class distributions
plt.subplot(2, 1, 2)
x = np.arange(len(unique_classes))
width = 0.25

plt.bar(x - width, original_class_dist, width, label='Original Dataset', color='green')
plt.bar(x, train_class_dist, width, label='Training Set', color='blue')
plt.bar(x + width, test_class_dist, width, label='Test Set', color='orange')

plt.xlabel('Class')
plt.ylabel('Proportion')
plt.title('Class Distribution After Stratified Shuffle Split')
plt.xticks(x, [f'Class {i}' for i in unique_classes])
plt.legend()

plt.tight_layout()
plt.show()

# Let's evaluate a model with both strategies
print("\nComparison of model performance with different sampling strategies:")

# Regular train-test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42
)

# Stratified train-test split
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=42
)

# Train a model on both splits
model = LogisticRegression(class_weight='balanced')

model.fit(X_train_reg, y_train_reg)
y_pred_reg = model.predict(X_test_reg)
acc_reg = accuracy_score(y_test_reg, y_pred_reg)

model.fit(X_train_strat, y_train_strat)
y_pred_strat = model.predict(X_test_strat)
acc_strat = accuracy_score(y_test_strat, y_pred_strat)

print(f"Regular Split Accuracy: {acc_reg:.4f}")
print(f"Stratified Split Accuracy: {acc_strat:.4f}")

## Time Series Splitting

For time series data, standard random splits can lead to data leakage because future data might be used to predict past events. The `TimeSeriesSplit` class helps maintain the temporal order in cross-validation.

In [ ]:
# Generate synthetic time series data
np.random.seed(42)
n_samples = 100
time = np.arange(n_samples)
trend = 0.1 * time
season = 10 * np.sin(2 * np.pi * time / 12)  # Yearly seasonality
noise = np.random.normal(0, 2, n_samples)
y_time = trend + season + noise

# Create features (lagged variables)
X_time = np.column_stack([
    y_time[:-1],  # t-1
    np.roll(y_time, 2)[:-1],  # t-2
    np.roll(y_time, 3)[:-1]   # t-3
])
y_time = y_time[1:]  # Target is the next value

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Visualize the TimeSeriesSplit
plt.figure(figsize=(15, 10))
plt.subplot(2, 1, 1)

# Plot the time series data
plt.plot(time, trend + season, 'g-', alpha=0.7, label='Signal')
plt.plot(time, trend + season + noise, 'b-', alpha=0.5, label='Signal + Noise')
plt.legend()
plt.title('Synthetic Time Series Data')
plt.xlabel('Time')
plt.ylabel('Value')

# Plot the time series cross-validation splits
plt.subplot(2, 1, 2)
fold_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, (train_index, test_index) in enumerate(tscv.split(X_time)):
    fold_color = fold_colors[i % len(fold_colors)]
    
    # Shift indices by 1 to align with original time series
    train_index = [idx + 1 for idx in train_index]
    test_index = [idx + 1 for idx in test_index]
    
    # Plot training data for this fold
    plt.scatter(train_index, [i + 0.5] * len(train_index),
                c=fold_color, s=10, label=f'Fold {i+1} Train')
    
    # Plot testing data for this fold
    plt.scatter(test_index, [i + 0.5] * len(test_index),
                c=fold_color, s=40, marker='X', label=f'Fold {i+1} Test')

plt.yticks(np.arange(5) + 0.5, [f'Fold {i+1}' for i in range(5)])
plt.title('Time Series Cross-Validation Splits')
plt.xlabel('Sample index (time)')

# Create a custom legend
handles, labels = [], []
for i in range(5):
    fold_color = fold_colors[i % len(fold_colors)]
    handles.append(plt.Line2D([0], [0], marker='o', color='w', 
                            markerfacecolor=fold_color, markersize=10))
    handles.append(plt.Line2D([0], [0], marker='X', color='w', 
                            markerfacecolor=fold_color, markersize=10))
    labels.append(f'Fold {i+1} Train')
    labels.append(f'Fold {i+1} Test')

plt.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15),
           ncol=5, fancybox=True)
           
plt.tight_layout()
plt.show()

# Compare regular CV with time series CV
print("Time series cross-validation model performance:")

# Train and evaluate with TimeSeriesSplit
model = LinearRegression()
tscv_scores = cross_val_score(
    model, X_time, y_time, 
    cv=tscv, 
    scoring='neg_mean_squared_error'
)
tscv_rmse = np.sqrt(-tscv_scores)

# Train and evaluate with regular KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(
    model, X_time, y_time, 
    cv=kf, 
    scoring='neg_mean_squared_error'
)
kf_rmse = np.sqrt(-kf_scores)

# Print the results
print(f"TimeSeriesSplit RMSE: {np.mean(tscv_rmse):.4f} ± {np.std(tscv_rmse):.4f}")
print(f"KFold (shuffled) RMSE: {np.mean(kf_rmse):.4f} ± {np.std(kf_rmse):.4f}")

# Visualize the results
plt.figure(figsize=(10, 6))

plt.boxplot([tscv_rmse, kf_rmse], 
           labels=['TimeSeriesSplit', 'KFold (shuffled)'],
           patch_artist=True,
           boxprops=dict(facecolor='lightblue'))

plt.scatter([1] * len(tscv_rmse), tscv_rmse, color='blue', alpha=0.7)
plt.scatter([2] * len(kf_rmse), kf_rmse, color='blue', alpha=0.7)

plt.ylabel('Root Mean Squared Error')
plt.title('Comparing Time Series CV with Regular CV')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("\nNote: Lower RMSE with regular KFold suggests data leakage - ")
print("the model is using 'future' information to predict the 'past'!")

## Data Leakage Prevention

Data leakage occurs when information from outside the training dataset is used to create the model. It's a critical issue that can lead to overly optimistic performance estimates and poor generalization.

Common sources of data leakage:

1. Preprocessing data before splitting it
2. Including target-correlated features that wouldn't be available during prediction
3. Using future information in time series forecasting
4. Including identifiers or duplicated samples across splits

Let's explore some examples and prevention strategies:

In [ ]:
# Example: Data leakage from preprocessing before splitting

# Load the breast cancer dataset
X, y = load_breast_cancer(return_X_y=True)

# Method 1 (INCORRECT): Scale the data before splitting
print("Method 1 (INCORRECT): Scale before splitting")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the pre-scaled data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Train a model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_incorrect = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy_incorrect:.4f}")

# Method 2 (CORRECT): Split first, then scale using only training data
print("\nMethod 2 (CORRECT): Split first, then scale")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scale using only training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)  # Use the scaler fitted on training data

# Train a model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy_correct = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy_correct:.4f}")

# Method 3 (CORRECT ALTERNATIVE): Use pipelines to prevent leakage
print("\nMethod 3 (CORRECT ALTERNATIVE): Use scikit-learn Pipeline")
# Split the raw data
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Create a pipeline that includes scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train the pipeline
pipeline.fit(X_train_raw, y_train)
y_pred = pipeline.predict(X_test_raw)
accuracy_pipeline = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy_pipeline:.4f}")

# Compare the results
print("\nComparison of methods:")
print(f"Method 1 (INCORRECT): {accuracy_incorrect:.4f}")
print(f"Method 2 (CORRECT): {accuracy_correct:.4f}")
print(f"Method 3 (PIPELINE): {accuracy_pipeline:.4f}")

if accuracy_incorrect > accuracy_correct:
    print("\n⚠️ The incorrect method shows artificially higher performance!")
    print("This is a classic example of data leakage.")

### Preventing Data Leakage in Cross-Validation

We need to be particularly careful with data leakage in cross-validation. Here's how to properly use preprocessing within cross-validation:

In [ ]:
# Example: Proper preprocessing in cross-validation
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import cross_val_score

# Load breast cancer dataset again
X, y = load_breast_cancer(return_X_y=True)

# Method 1 (INCORRECT): Preprocess then cross-validate
print("Method 1 (INCORRECT): Preprocess then cross-validate")
# Apply feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Generate polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)

# Perform cross-validation on pre-processed data
incorrect_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_poly, y,
    cv=5,
    scoring='accuracy'
)

print(f"Mean CV accuracy: {np.mean(incorrect_scores):.4f}")

# Method 2 (CORRECT): Use Pipeline with cross-validation
print("\nMethod 2 (CORRECT): Use Pipeline with cross-validation")
# Create a pipeline that includes preprocessing steps
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Perform cross-validation on raw data using the pipeline
correct_scores = cross_val_score(
    pipeline, X, y,
    cv=5,
    scoring='accuracy'
)

print(f"Mean CV accuracy: {np.mean(correct_scores):.4f}")

# Compare results
print("\nComparison:")
print(f"Method 1 (INCORRECT): {np.mean(incorrect_scores):.4f} ± {np.std(incorrect_scores):.4f}")
print(f"Method 2 (CORRECT): {np.mean(correct_scores):.4f} ± {np.std(correct_scores):.4f}")

if np.mean(incorrect_scores) > np.mean(correct_scores):
    print("\n⚠️ The incorrect method shows artificially higher performance!")
    print("This illustrates how data leakage can lead to overly optimistic performance estimates.")

### Tips to Prevent Data Leakage

1. **Always split data before preprocessing**: Train-test split should be the very first step
2. **Use pipelines**: Let scikit-learn's `Pipeline` handle the preprocessing steps
3. **Careful with time-series data**: Use specialized time-series cross-validation
4. **Check for target leakage**: Remove features that wouldn't be available during prediction
5. **Watch for duplicates**: Ensure that similar examples don't appear in both train and test sets
6. **Group data properly**: When data has natural groupings, split by groups
7. **Validate your validation strategy**: Check if your performance metrics are realistic

## Practical Examples

Let's put all this together with some complete end-to-end examples on real datasets.

In [ ]:
# Example 1: Classification problem (Breast Cancer)
# We'll compare different splitting strategies on a real world dataset

# Load the dataset
breast_cancer = load_breast_cancer()
X, y = breast_cancer.data, breast_cancer.target

# Create our model pipeline
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Define our evaluation strategies
cv_strategies = {
    'Simple Train-Test Split': lambda: train_test_split(X, y, test_size=0.3, random_state=42),
    'Stratified Train-Test Split': lambda: train_test_split(X, y, test_size=0.3, random_state=42, stratify=y),
    'K-Fold Cross-Validation': KFold(n_splits=5, shuffle=True, random_state=42),
    'Stratified K-Fold CV': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
}

# Function to evaluate with simple train-test splits
def evaluate_split(split_func, model):
    X_train, X_test, y_train, y_test = split_func()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Results dictionary
results = {}

# Evaluate train-test splits
for name, split_func in list(cv_strategies.items())[:2]:
    accuracy = evaluate_split(split_func, model_pipeline)
    results[name] = [accuracy]  # Wrap in list for consistency with CV results
    print(f"{name}: Accuracy = {accuracy:.4f}")

# Evaluate cross-validation strategies
for name, cv in list(cv_strategies.items())[2:]:
    scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name}: Mean Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# Visualize results
plt.figure(figsize=(12, 6))
data = [results[name] for name in cv_strategies.keys()]
box = plt.boxplot(data, patch_artist=True, labels=list(cv_strategies.keys()))

# Color the boxes
colors = ['lightblue', 'lightgreen', 'lightsalmon', 'lightpink']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

# Add individual points where applicable
for i, name in enumerate(cv_strategies.keys()):
    x_pos = i + 1
    if len(results[name]) > 1:  # Only for CV strategies with multiple scores
        plt.scatter([x_pos] * len(results[name]), results[name], 
                    alpha=0.7, s=30, color='darkblue')
    
    # Show mean for all
    plt.text(x_pos, min(results[name])-0.01, 
             f"μ={np.mean(results[name]):.3f}", 
             ha='center', fontweight='bold')

plt.title('Comparison of Different Evaluation Strategies on Breast Cancer Dataset')
plt.ylabel('Accuracy Score')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Example 2: Regression problem with feature engineering
# We'll build a more complex pipeline and evaluate it properly

# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

# Define our pipeline with more preprocessing steps
regression_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('feature_selection', SelectFromModel(LinearRegression())),
    ('regressor', Ridge(alpha=1.0))
])

# Evaluation strategies
regression_cv_strategies = {
    'Simple Train-Test Split': lambda: train_test_split(X, y, test_size=0.3, random_state=42),
    'KFold CV': KFold(n_splits=5, shuffle=True, random_state=42),
    'TimeSeriesSplit': TimeSeriesSplit(n_splits=5)
}

# Function to evaluate regression models
def evaluate_regression_split(split_func, model):
    X_train, X_test, y_train, y_test = split_func()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return np.sqrt(mse), r2  # Return RMSE and R²

# Results dictionary
regression_results = {'rmse': {}, 'r2': {}}

# Evaluate train-test split
name, split_func = list(regression_cv_strategies.items())[0]
rmse, r2 = evaluate_regression_split(split_func, regression_pipeline)
regression_results['rmse'][name] = [rmse]
regression_results['r2'][name] = [r2]
print(f"{name}: RMSE = {rmse:.2f}, R² = {r2:.4f}")

# Function to calculate RMSE from CV scores
def rmse_from_cv(scores):
    return np.sqrt(-scores)

# Evaluate cross-validation strategies
for name, cv in list(regression_cv_strategies.items())[1:]:
    # Calculate MSE scores (negative, as sklearn uses scoring where higher is better)
    mse_scores = cross_val_score(regression_pipeline, X, y, cv=cv, 
                                scoring='neg_mean_squared_error')
    rmse_scores = rmse_from_cv(mse_scores)
    
    # Calculate R² scores
    r2_scores = cross_val_score(regression_pipeline, X, y, cv=cv, 
                               scoring='r2')
    
    regression_results['rmse'][name] = rmse_scores
    regression_results['r2'][name] = r2_scores
    
    print(f"{name}: Mean RMSE = {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}, "
          f"Mean R² = {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

# Create visualization for RMSE
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
rmse_data = [regression_results['rmse'][name] for name in regression_cv_strategies.keys()]
rmse_box = plt.boxplot(rmse_data, patch_artist=True, labels=list(regression_cv_strategies.keys()))

# Color the boxes
colors = ['lightblue', 'lightgreen', 'lightsalmon']
for patch, color in zip(rmse_box['boxes'], colors):
    patch.set_facecolor(color)

# Add individual points where applicable
for i, name in enumerate(regression_cv_strategies.keys()):
    x_pos = i + 1
    if len(regression_results['rmse'][name]) > 1:  # Only for CV strategies with multiple scores
        plt.scatter([x_pos] * len(regression_results['rmse'][name]), 
                   regression_results['rmse'][name],
                   alpha=0.7, s=30, color='darkblue')
    
    # Show mean for all
    plt.text(x_pos, min(regression_results['rmse'][name])-5, 
             f"μ={np.mean(regression_results['rmse'][name]):.1f}", 
             ha='center', fontweight='bold')

plt.title('RMSE by Evaluation Strategy (Diabetes Dataset)')
plt.ylabel('Root Mean Squared Error (RMSE)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Create visualization for R²
plt.subplot(1, 2, 2)
r2_data = [regression_results['r2'][name] for name in regression_cv_strategies.keys()]
r2_box = plt.boxplot(r2_data, patch_artist=True, labels=list(regression_cv_strategies.keys()))

# Color the boxes
for patch, color in zip(r2_box['boxes'], colors):
    patch.set_facecolor(color)

# Add individual points where applicable
for i, name in enumerate(regression_cv_strategies.keys()):
    x_pos = i + 1
    if len(regression_results['r2'][name]) > 1:  # Only for CV strategies with multiple scores
        plt.scatter([x_pos] * len(regression_results['r2'][name]), 
                   regression_results['r2'][name],
                   alpha=0.7, s=30, color='darkblue')
    
    # Show mean for all
    plt.text(x_pos, min(regression_results['r2'][name])-0.05, 
             f"μ={np.mean(regression_results['r2'][name]):.2f}", 
             ha='center', fontweight='bold')

plt.title('R² by Evaluation Strategy (Diabetes Dataset)')
plt.ylabel('R² Score')
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Summary and Best Practices

In this notebook, we explored various techniques for splitting data in machine learning:

### Key Concepts:
1. **Basic train-test splitting**: Simple but effective for large datasets
2. **Cross-validation**: Provides more robust performance estimates
3. **Stratified sampling**: Maintains class distribution in imbalanced datasets
4. **Time series splitting**: Respects temporal order in sequential data
5. **Data leakage prevention**: Critical for realistic performance estimates

### Best Practices:

1. **Choose the right splitting strategy** based on your data:
   - Standard classification/regression: Stratified K-Fold CV
   - Imbalanced classes: Stratified sampling
   - Time series: TimeSeriesSplit
   - Large datasets: Simple train-test split may be sufficient

2. **Prevent data leakage**:
   - Always split data before preprocessing
   - Use scikit-learn's Pipeline to ensure proper preprocessing within CV
   - Be cautious with feature engineering

3. **Evaluate thoroughly**:
   - Use multiple metrics appropriate for your problem
   - Consider the variance in performance across folds
   - Be skeptical of results that seem too good to be true

4. **Document your approach**:
   - Record random seeds used for reproducibility
   - Note any special handling for particular features

By following these practices, you'll build more robust models with more reliable performance estimates.